In [ ]:
#imports and sets up the database connection
import os
import pandas as pd
from pathlib import Path
from sqlalchemy import create_engine
from dotenv import load_dotenv

load_dotenv(Path("../.env"))
engine = create_engine(
    f"postgresql+psycopg2://{os.getenv('DB_USER')}:{os.getenv('DB_PASSWORD')}"
    f"@{os.getenv('DB_HOST')}:{os.getenv('DB_PORT')}/{os.getenv('DB_NAME')}"
)

def q(sql):
    return pd.read_sql(sql, engine)

In [ ]:
#Overall KPIs
q("""
SELECT COUNT(*) AS total_accidents,
       SUM(casualties) AS total_casualties,
       ROUND(AVG(risk_score)::numeric,2) AS avg_risk,
       ROUND(AVG(casualties)::numeric,2) AS avg_casualties,
       SUM(CASE WHEN is_fatal THEN 1 ELSE 0 END) AS fatal_accidents
FROM accidents;
""")

,total_accidents,total_casualties,avg_risk,avg_casualties,fatal_accidents
0,20000,34529,0.44,1.73,2987


In [ ]:
#Severity distribution
q("""
SELECT accident_severity, COUNT(*) AS cnt,
       ROUND(100.0*COUNT(*)/SUM(COUNT(*)) OVER (), 2) AS pct
FROM accidents
GROUP BY accident_severity
ORDER BY cnt DESC;
""")

,accident_severity,cnt,pct
0,Minor,11025,55.13
1,Major,5988,29.94
2,Fatal,2987,14.94


In [ ]:

#Weekend vs Weekday
q("""
SELECT is_weekend, COUNT(*) AS accidents,
       ROUND(AVG(risk_score)::numeric,2) AS avg_risk,
       SUM(casualties) AS casualties
FROM accidents
GROUP BY is_weekend;
""")

,is_weekend,accidents,avg_risk,casualties
0,False,14277,0.44,24636
1,True,5723,0.44,9893


In [5]:
#Hourly trend
q("""
SELECT hour, COUNT(*) AS accidents,
       ROUND(AVG(risk_score)::numeric,2) AS avg_risk
FROM accidents
GROUP BY hour
ORDER BY hour;
""")

,hour,accidents,avg_risk
0,0,840,0.36
1,1,859,0.35
2,2,888,0.34
3,3,824,0.35
4,4,805,0.36
5,5,827,0.41
6,6,856,0.39
7,7,786,0.40
8,8,849,0.61
9,9,786,0.59


In [6]:
#Day of week trend
q("""
SELECT day_of_week, COUNT(*) AS accidents,
       ROUND(AVG(risk_score)::numeric,2) AS avg_risk
FROM accidents
GROUP BY day_of_week
ORDER BY accidents DESC;
""")

,day_of_week,accidents,avg_risk
0,Monday,2966,0.44
1,Friday,2879,0.43
2,Tuesday,2871,0.44
3,Saturday,2867,0.44
4,Sunday,2856,0.44
5,Thursday,2835,0.44
6,Wednesday,2726,0.43


In [7]:
#Monthly trend
q("""
SELECT month, month_name, COUNT(*) AS accidents
FROM accidents
GROUP BY month, month_name
ORDER BY month;
""")

,month,month_name,accidents
0,1,Jan,2044
1,2,Feb,1865
2,3,Mar,2043
3,4,Apr,1763
4,5,May,1593
5,6,Jun,1512
6,7,Jul,1543
7,8,Aug,1588
8,9,Sep,1475
9,10,Oct,1516


In [8]:
#Top states
q("""
SELECT state, COUNT(*) AS accidents,
       SUM(casualties) AS casualties,
       ROUND(AVG(risk_score)::numeric,2) AS avg_risk
FROM accidents
GROUP BY state
ORDER BY accidents DESC;
""")

,state,accidents,casualties,avg_risk
0,Maharashtra,5009,8716,0.44
1,Punjab,2577,4377,0.44
2,Tamil Nadu,2575,4567,0.44
3,West Bengal,2559,4508,0.43
4,Karnataka,2438,4128,0.44
5,Delhi,2433,4175,0.44
6,Telangana,2409,4058,0.44


In [9]:
#Top cities
q("""
SELECT city, state, COUNT(*) AS accidents, SUM(casualties) AS casualties
FROM accidents
GROUP BY city, state
ORDER BY accidents DESC;
""")

,city,state,accidents,casualties
0,Chandigarh,Punjab,2577,4377
1,Chennai,Tamil Nadu,2575,4567
2,Kolkata,West Bengal,2559,4508
3,Pune,Maharashtra,2517,4419
4,Mumbai,Maharashtra,2492,4297
5,Bangalore,Karnataka,2438,4128
6,Delhi,Delhi,2433,4175
7,Hyderabad,Telangana,2409,4058


In [10]:
#Weather conditions
q("""
SELECT weather, COUNT(*) AS accidents,
       ROUND(AVG(risk_score)::numeric,2) AS avg_risk
FROM accidents
GROUP BY weather
ORDER BY accidents DESC;
""")

,weather,accidents,avg_risk
0,Clear,6690,0.24
1,Rain,6677,0.49
2,Fog,6633,0.59


In [11]:
#Visibility conditions
q("""
SELECT visibility, COUNT(*) AS accidents,
       ROUND(AVG(risk_score)::numeric,2) AS avg_risk
FROM accidents
GROUP BY visibility
ORDER BY accidents DESC;
""")

,visibility,accidents,avg_risk
0,Low,9987,0.59
1,High,6690,0.24
2,Medium,3323,0.39


In [12]:
#Cause of accident
q("""
SELECT cause, COUNT(*) AS accidents,
       SUM(casualties) AS total_casualties,
       ROUND(AVG(risk_score)::numeric,2) AS avg_risk
FROM accidents
GROUP BY cause
ORDER BY accidents DESC;
""")

,cause,accidents,total_casualties,avg_risk
0,Distraction,4026,6856,0.43
1,Overspeeding,4025,6943,0.44
2,Weather,3997,6953,0.44
3,Drunk Driving,3978,6869,0.44
4,Poor Road,3974,6908,0.44


In [13]:
#Road type
q("""
SELECT road_type, COUNT(*) AS accidents,
       ROUND(AVG(casualties)::numeric,2) AS avg_casualties
FROM accidents
GROUP BY road_type
ORDER BY accidents DESC;
""")

,road_type,accidents,avg_casualties
0,Urban,6745,1.73
1,Rural,6639,1.72
2,Highway,6616,1.73


In [14]:
#Festival impact
q("""
SELECT is_festival, COUNT(*) AS accidents,
       ROUND(AVG(risk_score)::numeric,2) AS avg_risk,
       SUM(casualties) AS casualties
FROM accidents
GROUP BY is_festival;
""")

,is_festival,accidents,avg_risk,casualties
0,False,19885,0.44,34295
1,True,115,0.56,234


In [16]:
#Traffic density vs accident severity
q("""
SELECT traffic_density, accident_severity, COUNT(*) AS cnt
FROM accidents
GROUP BY traffic_density, accident_severity
ORDER BY traffic_density, cnt DESC;
""")

,traffic_density,accident_severity,cnt
0,High,Minor,3912
1,High,Major,2077
2,High,Fatal,1045
3,Low,Minor,3890
4,Low,Major,2116
5,Low,Fatal,1061
6,Medium,Minor,3223
7,Medium,Major,1795
8,Medium,Fatal,881
